# STAT 207 Homework 11 [25 points]

## Regularization Models for Linear Relationships

Due: Monday, May 4, end of day (11:59 pm CT)

Late submissions accepted until Tuesday, May 5 at noon

<hr>

## Imports 

Run the following code cell to import the necessary packages into the file.  You may import additional packages, as needed for this assignment.

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

## The Data

With available climate data dating back many decades and the prevalence of climate change, humans are looking to understand exactly how different features of the climate affect the temperatures globally.  For this assignment, we will look to understand how the **global temperature** fluctuates based on other environmental features.

We will use various atmospheric and temperature measures over 309 months from 1983 to 2008, with the following variables:

- **`Year`**: the observation year
- **`Month`**: the observation month, recorded with numbers 1 to 12
- **`MEI`**: Multivariate El Nino Southern Oscillation Index (MEI), measuring the affects of the El Nino weather pattern
- **`CO2`**: atmospheric concentration of carbon dioxide (in ppmv, parts per million by volume)
- **`CH4`**: atmospheric concentration of methane (in ppmv)
- **`N2O`**: atmospheric concentration of nitrous oxide (in ppmv)
- **`CFC-11`**: atmospheric concentration of CCl3F or trichlorofluoromethane (in ppbv, parts per billion by volume)
- **`CFC-12`**: atmospheric concentration of CCl2F2 or dichlorodifluoromethane (in ppbv)
- **`TSI`**: the total solar irradiance (TSI) (in W/m2), measuring the rate at which the sun's energy is deposited per unit area.
- **`Aerosols`**: the mean stratospheric aerosol optical depth at 500 nm, a measure associated with volcanic activity
- **`Temp`**: the difference in the average global temperature for that month (in Celsius) and a reference value

The ESRL/NOAA Physical Sciences Division reports the MEI; atmospheric concentrations are measured by the ESRL/NOAA Global Monitoring Division; the SOLARIS-HEPPA project website provides the TSI; the Godard Institute for Space Studies at NASA reports the Aerosols; and the Climatic Research Unit at the University of East Anglia reports the Temp.

Run the code in the cell below to read in the cleaned data for this document.  The data is saved as `df` with this code.  

In [31]:
df = pd.read_csv('climate_change.csv')
df_train = df[df['Year'] <= 2006]
df_test = df[df['Year'] >= 2007]
X_train = df_train.drop(['Year', 'Month', 'Temp'], axis = 1)
X_test = df_test.drop(['Year', 'Month', 'Temp'], axis = 1)
y_train = df_train['Temp']
y_test = df_test['Temp']

## 1. Summarize Data [1.5 points]

Above, we set aside a training data.  We didn't randomly select our training and test set; instead, imagine that we fit a model using the available data in 2006 in our training data.  We'll then use the data that we collect in the following two years as the test set to evaluate this model.

As defined in our X_train and X_test above, our response variable for this assignment with be **`Temp`**.  We'll use all variables except the **`Year`** and **`Month`** as our predictor variables.

**a)** Scale our predictor variables in the training data.

In [32]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_train = scaler.fit_transform(X_train)

X_train = pd.DataFrame(scaled_train, columns=X_train.columns)
X_train.head()

,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
0,2.385858,-1.353318,-2.352555,-1.680002,-2.919383,-2.444829,0.002404,2.288937
1,1.966677,-1.391849,-2.459597,-1.665476,-2.884396,-2.415393,0.048338,2.058639
2,1.507626,-1.511818,-2.470345,-1.655161,-2.848073,-2.383549,0.458248,1.848367
3,0.849221,-1.678200,-2.511363,-1.645898,-2.810652,-2.351178,0.795762,1.654783
4,0.092756,-1.860344,-2.137373,-1.632846,-2.772944,-2.320097,0.329683,1.474550


**b)** Now, we want to be sure that we also scale our test data, using the same scaling as applied to our training data.  Apply your scaling algorithm from **part a** to the test data.  

*Note*: This does not include re-fitting your scaling process.  You will re-use your scaling process from **part a**, simply transforming your test data with the same scaling process.

In [33]:
scaled_test = scaler.fit_transform(X_test)

X_test = pd.DataFrame(scaled_train, columns=X_test.columns)
X_test.head()

,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
0,2.385858,-1.353318,-2.352555,-1.680002,-2.919383,-2.444829,0.002404,2.288937
1,1.966677,-1.391849,-2.459597,-1.665476,-2.884396,-2.415393,0.048338,2.058639
2,1.507626,-1.511818,-2.470345,-1.655161,-2.848073,-2.383549,0.458248,1.848367
3,0.849221,-1.678200,-2.511363,-1.645898,-2.810652,-2.351178,0.795762,1.654783
4,0.092756,-1.860344,-2.137373,-1.632846,-2.772944,-2.320097,0.329683,1.474550


**c)** I'd suggest turning to Q1 on Gradescope here.

You'll likely need to further explore the data for Gradescope Q1.  You can use this section for any exploration that you'd like, although there are no points associated with this part.

In [34]:
X_train.describe()

,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
count,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,2.840000e+02,284.000000
mean,3.752867e-17,2.501911e-15,2.301758e-15,-2.001529e-16,-1.150879e-15,2.501911e-16,-4.336813e-13,0.000000
std,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765e+00,1.001765
min,-2.077501e+00,-1.860344e+00,-2.543388e+00,-1.680002e+00,-2.919383e+00,-2.444829e+00,-1.685916e+00,-0.538056
25%,-7.165109e-01,-7.968154e-01,-6.469486e-01,-8.421301e-01,-1.398208e-01,-5.373796e-01,-8.659712e-01,-0.501341
50%,-3.601564e-02,-1.334784e-01,2.799670e-01,-1.700429e-01,3.764272e-01,4.728576e-01,-1.171738e-01,-0.384524
75%,5.992210e-01,8.030748e-01,7.851835e-01,9.414536e-01,7.141010e-01,7.932329e-01,7.435250e-01,-0.124187
max,2.865383e+00,2.063634e+00,1.366734e+00,1.851271e+00,9.072212e-01,8.414196e-01,3.032543e+00,4.394997


## 2. Fitting A Model [1 point]

**a)** Fit a LASSO model with $\lambda = 0.06$ to the training data, including all variables except the year and month variables, as set up in the provided $X$ features matrix.  Print the coefficients for this model.

In [35]:
lasso_mod_06 = Lasso(alpha=0.06, max_iter=1000)
lasso_mod_06.fit(X_train, y_train)
df_slopes = pd.DataFrame(lasso_mod_06.coef_.T, columns=["lasso_mod_06"], index=X_train.columns)
df_slopes

,lasso_mod_06
MEI,0.000000
CO2,0.079893
CH4,0.000000
N2O,0.002758
CFC-11,0.000000
CFC-12,0.000000
TSI,0.000000
Aerosols,-0.000000


**b)** I'd suggest turning to Q2 on Gradescope here.

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like, although there are no points associated with this part.

## 3. Picking a Best Model [2.5 points]

Instead of using a LASSO model, we decide that we'd rather move forward with a **ridge regression** model.  We don't know which $\lambda$ to use for this model, so let's explore which value of $\lambda$ might be most appropriate for a ridge regression model.

To do this, we'll explore $\lambda$ values between 0.05 and 1 exploring by every 0.05.  We can do this with code using:

`for m in range(1, 21):`

`    alph = m / 20`

The following code sets up the folds for cross-validation.

In [36]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

cross_val = KFold(n_splits = 10, shuffle = True, random_state = 202605)

**a)** Use 10-fold cross-validation to explore the $R^2$ values for each of the $\lambda$s as defined above.

In [37]:
lambdas = []
mean_r2s = []

for i in range(1, 21):
    alph = i / 20
    ridge_mod = Ridge(alpha=alph, max_iter=1000)
    test_fold_r2 = cross_val_score(ridge_mod, X_train, y_train, cv=cross_val, scoring="r2")
    lambdas.append(alph)
    mean_r2s.append(test_fold_r2.mean())
df_cv_results = pd.DataFrame({'lambda': lambdas, 'mean_test_R^2': mean_r2s})

**b)** Print the $R^2$ values for each of the individual folds of your optimal $\lambda$.

In [38]:
test_fold_r2

array([0.72184293, 0.74653213, 0.7755094 , 0.75973793, 0.74833901,
       0.77156001, 0.784429  , 0.43366909, 0.68042044, 0.72603696])

**c)** Repeat this process for a different set of 10-folds, using a random state of your choosing.  Determine which $\lambda$ results in the optimal mean $R^2$, similar to what you did above.

In [39]:
cross_val_2 = KFold(n_splits=10, shuffle=True, random_state=42)
lambdas_2 = []
mean_r2s_2 = []

for i in range(1, 21):
    alph = i / 20
    ridge_mod = Ridge(alpha=alph, max_iter=1000)
    test_fold_r2 = cross_val_score(ridge_mod, X_train, y_train, cv=cross_val_2, scoring="r2")
    lambdas_2.append(alph)
    mean_r2s_2.append(test_fold_r2.mean())

df_cv_results_2 = pd.DataFrame({'lambda': lambdas_2, 'mean_test_R^2': mean_r2s_2})

**d)** I'd suggest turning to Q3 on Gradescope here.

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like, although there are no points associated with this part.

## 4. Evaluating Our Best Model [1 point]

**a)** Refit the ridge regression model with the optimal $\lambda$ found in Question **3b**, but this time fit the model to the full training data.  Print the resulting coefficients.

In [40]:
ridge_mod = Ridge(alpha=0.1, max_iter=1000)
ridge_mod.fit(X_train, y_train)
ridge_mod.coef_

array([ 0.05952399,  0.07336284,  0.00614979, -0.07103694, -0.1315225 ,
        0.21136885,  0.03707498, -0.04612572])

**b)** You'll be asked to calculate the $R^2$ on the test data for the model from part **a** above.  

You will need to perform additional calculations or analyses to do this.  This space is available for those calculations, although there are no points associated with the code.  You will enter the results of your calculation on **Q4.2** on Gradescope.

In [ ]:
ridge_mod.score(X_test, y_test)

## 5. AI Acknowledgement

Our course policy is that you should write all of your own interpretations and other narrative answers (phrases or sentences) yourself without the assistance of AI.  You may use AI to help guide your code, although you should write all of your own code yourself (not copy-paste from another source) and you should cite your use of AI.  I would encourage you to try to generate any necessary code yourself first using course resources and using AI as a debugging tool if/when you reach an error that you can't figure out or to help you perform any coding tasks that are more advanced than we've demonstrated during class (intended only for projects).  

Did you use AI on this assignment?  Did you use other resources outside of our course-provided resources on this assignment?

no

If you used AI or other resources, answer the following questions to cite your usage.

- Which AI and/or resources did you use (including links, if appropriate)?
- What prompts did you ask it?
- How did you integrate the responses into your assignment?  Specifically, which questions or parts are associated with this usage?

Note: answering these three questions are enough for our course but may not be enough for a different course or context.

Remember to keep all your cells and hit the save icon above periodically to checkpoint (save) your results on your local computer. Once you are satisified with your results restart the kernel and run all (Kernel -> Restart & Run All). **Make sure nothing has changed**. Checkpoint and exit (File -> Save and Checkpoint + File -> Close and Halt). Follow the instructions on the Homework 11 Canvas Assignment to submit your notebook to GitHub.